In [this notebook](https://decoupler-py.readthedocs.io/en/latest/notebooks/dorothea.html) we show how to use [decoupler](https://decoupler-py.readthedocs.io/en/latest/) for TF activity inference with only the gene expression matrix. 

In [1]:
import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

from anndata import AnnData

## Loading CollecTRI network
CollecTRI is a comprehensive resource containing a curated collection of TFs and their transcriptional targets compiled from 12 different resources.

In [6]:
net = dc.get_collectri(organism='human', split_complexes=False)

## Applying to lymphnote

In [ ]:
adata = sc.read_h5ad("lymphnode/results_lymphnode/lymphnode.h5ad")

### Activity inference with univariate linear model (ULM)
For each cell in `adata` and each TF in `net`, it fits a linear model that predicts the observed gene expression based solely on the TF-Gene interaction weights. Once fitted, the obtained $t$-value of the slope is the score. If it is positive, we interpret that the TF is active and if it is negative we interpret that it is inactive.

In [7]:
dc.run_ulm(
    mat=adata,
    net=net,
    source='source',
    target='target',
    weight='weight',
    verbose=True,
    use_raw=False)

Running ulm on mat with 3989 samples and 21261 targets for 725 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.34it/s]


The obtained scores `ulm_estimate` and $p$-values `ulm_pvals` are stored in the `.obsm` key.

In [8]:
adata.obsm['ulm_estimate']

,ABL1,AEBP1,AHR,AHRR,AIP,AIRE,AP1,APEX1,AR,ARID1A,...,ZNF382,ZNF384,ZNF395,ZNF423,ZNF436,ZNF699,ZNF76,ZNF804A,ZNF91,ZXDC
AAACAAGTATCTCCCA-1,0.100334,0.220511,0.034902,0.033616,0.018800,0.010221,0.737065,0.912426,1.569138,-0.102717,...,-0.168692,-0.050145,-0.020222,-0.131061,0.056706,0.049664,-0.049597,-0.055856,0.445793,0.730417
AAACAATCTACTAGCA-1,0.700104,1.343486,0.268150,-0.201587,0.227726,0.022344,2.576660,3.340938,3.283575,-0.082629,...,0.012134,-0.418828,-0.183535,-0.379586,0.330079,0.237282,0.126222,-0.223482,2.047299,2.912783
AAACACCAATAACTGC-1,0.428395,2.115787,0.436808,-0.096871,-0.030195,0.299115,2.791031,3.055013,3.164736,-0.130822,...,-0.081876,-0.374073,-0.085079,-0.324009,0.055133,0.053981,-0.082938,-0.108983,1.380370,1.666318
AAACAGAGCGACTCCT-1,0.260086,1.551232,0.650885,-0.184094,0.184214,0.110435,2.486469,3.510542,3.450200,-0.208242,...,-0.052799,-0.500119,0.288617,-0.394712,0.239756,0.085977,-0.022356,0.288823,2.176333,2.496749
AAACAGCTTTCAGAAG-1,0.204285,0.536005,0.225582,0.025112,-0.167475,-0.103865,2.142221,3.464632,2.338708,-0.023261,...,-0.279493,-0.398039,-0.201849,-0.335127,-0.114769,-0.166062,0.102606,0.060154,1.330444,2.367034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,0.227440,0.908130,0.109269,-0.095849,-0.061173,0.242019,1.930747,2.150409,2.273036,-0.015455,...,0.039122,-0.069381,-0.146057,-0.230204,0.202663,0.106787,0.009595,0.213711,0.955524,1.567372
TTGTTTCATTAGTCTA-1,0.596566,0.792820,0.535292,-0.053732,-0.050795,0.273192,2.400427,2.915891,2.633444,-0.046561,...,-0.351969,-0.244445,-0.273158,-0.268712,0.141134,0.216250,-0.046658,-0.134994,1.275963,1.641374
TTGTTTCCATACAACT-1,0.723144,1.850834,0.703964,-0.226408,0.307219,0.522238,4.074167,4.676816,3.714317,0.097484,...,-0.377766,-0.062755,-0.144683,-0.313637,0.390117,0.172577,-0.008093,-0.317118,2.070987,3.380344
TTGTTTGTATTACACG-1,0.132407,1.493822,0.352997,-0.435474,0.332507,-0.207450,2.308685,2.734928,2.949143,-0.155587,...,-0.053597,-0.440371,0.098887,-0.229536,0.059787,0.018368,-0.075222,-0.368042,1.477251,2.283289


In [9]:
adata.obsm['collectri_ulm_estimate'] = adata.obsm['ulm_estimate'].copy()
adata.obsm['collectri_ulm_pvals'] = adata.obsm['ulm_pvals'].copy()
adata

AnnData object with n_obs × n_vars = 3989 × 21261
    obs: 'in_tissue', 'array_row', 'array_col', 'germinal_center', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial', 'celltype', 'celltype_raw', 'ulm_estimate', 'ulm_pvals', 'collectri_ulm_estimate', 'collectri_ulm_pvals'

Extract the activity matrix.

In [10]:
acts = dc.get_acts(adata, obsm_key='ulm_estimate')
acts

C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 3989 × 725
    obs: 'in_tissue', 'array_row', 'array_col', 'germinal_center', 'n_counts'
    uns: 'spatial'
    obsm: 'spatial', 'celltype', 'celltype_raw', 'ulm_estimate', 'ulm_pvals', 'collectri_ulm_estimate', 'collectri_ulm_pvals'

In [11]:
adata.write("lymphnode/results_lymphnode/lymphnode_decoupler.h5ad")

## Applying to glioblastoma

In [14]:
adata = sc.read_h5ad("glioblastoma/results_glioblastoma/glioblastoma.h5ad")
dc.run_ulm(
    mat=adata,
    net=net,
    source='source',
    target='target',
    weight='weight',
    verbose=True,
    use_raw=False)

Running ulm on mat with 3462 samples and 20950 targets for 724 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.40it/s]


In [15]:
adata.obsm['collectri_ulm_estimate'] = adata.obsm['ulm_estimate'].copy()
adata.obsm['collectri_ulm_pvals'] = adata.obsm['ulm_pvals'].copy()
adata

AnnData object with n_obs × n_vars = 3462 × 20950
    obs: 'in_tissue', 'array_row', 'array_col', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial', 'ulm_estimate', 'ulm_pvals', 'collectri_ulm_estimate', 'collectri_ulm_pvals'

In [16]:
adata.write("glioblastoma/results_glioblastoma/glioblastoma_decoupler.h5ad")

## Applying to breast cancer

In [ ]:
!mkdir breast/results_breast/decoupler

In [18]:
for sample in ['1142243F', '1160920F', 'CID4290', 'CID4465', 'CID4535', 'CID44971']:
    adata = sc.read_h5ad("results_breast/breast_st_qc/{}.h5ad".format(sample))
    dc.run_ulm(
        mat=adata,
        net=net,
        source='source',
        target='target',
        weight='weight',
        verbose=True,
        use_raw=False)
    adata.obsm['collectri_ulm_estimate'] = adata.obsm['ulm_estimate'].copy()
    adata.obsm['collectri_ulm_pvals'] = adata.obsm['ulm_pvals'].copy()
    adata.write("breast/results_breast/decoupler/{}_decoupler.h5ad".format(sample))

C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3468 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
Running ulm on mat with 3462 samples and 20950 targets for 724 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.47it/s]
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3468 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
Running ulm on mat with 3462 samples and 20950 targets for 724 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.51it/s]
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.41it/s]
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3468 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
Running ulm on mat with 3462 samples and 20950 targets for 724 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.41it/s]
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\linan\anaconda3\envs\bioinfo\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3468 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
Running ulm on mat with 3462 samples and 20950 targets for 724 sources.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.42it/s]
